# Notebook Preprocesamiento y Pruebas de Modelos
Este notebook contiene el preprocesamiento propuesto para los datos a partir del EDA realizado. Este consta de:


Posterior a ello se realiza la prueba de diversos modelos, en los cuales se pretende maximizar el Recall debido a la necesidad de reducir los falsos negativos que podría resultar más peligroso para el paciente. 

## 0. Configuración

### A. Librerías

In [1]:
!pip install -q "monai[all]"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.2/55.2 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.1/51.1 kB 1.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 1.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 11.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 MB 22.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 MB 26.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.0/28.0 MB 63.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.2/57.2 MB 30.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28.5/28.5 MB 59.2 MB/

In [2]:
!pip install -q pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 37.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 44.6 MB/s eta 0:00:00:00:01


In [3]:
!pip install -q python-gdcm pylibjpeg pylibjpeg-libjpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 58.1 MB/s eta 0:00:00:00:01:01


In [4]:
import monai
print(monai.__version__)

1.6.1


Failed to load nvtiff library! Please install nvTIFF (see https://docs.nvidia.com/cuda/nvtiff/userguide.html#installing-nvtiff).
Note: If using nvImageCodec's Python distribution, it is enough to install the nvTIFF wheel: e.g. `python3 -m pip install nvidia-nvtiff-cu13`
[ERROR] [nvtiff_ext] Failed to load nvTIFF library or retrieve version. Decoders and encoders will not be registered.
Failed to load nvjpeg library! Please install CUDA toolkit or, if using nvImageCodec's Python distribution, the nvJPEG python wheel (e.g. python -m pip install nvidia-nvjpeg-cu13).
[ERROR] [nvjpeg-ext] Failed to load nvJPEG library or retrieve version. Decoders and encoders will not be registered.
Failed to load nvjpeg2k library! Please install nvJPEG2000 (see https://docs.nvidia.com/cuda/nvjpeg2000/userguide.html#installing-nvjpeg2000).
Note: If using nvImageCodec's Python distribution, it is enough to install the nvJPEG2000 wheel: e.g. `python3 -m pip install nvidia-nvjpeg2k-cu13`
[ERROR] [nvjpeg2k_ext

In [5]:
import os
import glob
import pandas as pd
import torch
import scipy.ndimage
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pydicom
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.data import CacheDataset, DataLoader, decollate_batch
from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    Resized,
    ScaleIntensityRangePercentilesd,
    AsDiscrete
)
from tqdm import tqdm

### B. Parámetros 

In [6]:
# Configuración visual
sns.set_theme(style="whitegrid")

# Rutas
base_dir = "/kaggle/input/competitions/rsna-2022-cervical-spine-fracture-detection"
path_train = "/kaggle/input/competitions/rsna-2022-cervical-spine-fracture-detection/train.csv"
image_dir = "/kaggle/input/competitions/rsna-2022-cervical-spine-fracture-detection/train_images"
output_dir = "/kaggle/working/processed_volumes"

### C. Funciones

#### Preprocesamiento

**Reordenar los cortes:** Esta función busca reordenar los cortes para que representen correctamente la columna cervical. 

In [7]:
# ==========================================
# PASO 1: EXTRACCIÓN Y ORDENAMIENTO ESPACIAL
# ==========================================
def obtener_rutas_y_espaciado(study_uid, image_dir):
    """
    Ordena los cortes por la coordenada Z real y calcula el espaciado físico original.
    """
    study_path = os.path.join(image_dir, study_uid)
    archivos_dcm = [f for f in os.listdir(study_path) if f.endswith('.dcm')]
    
    cortes = []
    pixel_spacing = None
    
    for archivo in archivos_dcm:
        ruta_completa = os.path.join(study_path, archivo)
        dcm = pydicom.dcmread(ruta_completa, stop_before_pixels=True)
        z_pos = float(dcm.ImagePositionPatient[2])
        
        if pixel_spacing is None and hasattr(dcm, 'PixelSpacing'):
            pixel_spacing = [float(dcm.PixelSpacing[0]), float(dcm.PixelSpacing[1])]
            
        cortes.append((z_pos, ruta_completa))
    
    # Ordenar por coordenada Z corrige el 90.24% de estudios invertidos
    cortes.sort(key=lambda x: x[0])
    rutas_ordenadas = [corte[1] for corte in cortes]
    
    # Calcular el espaciado Z real evitando la etiqueta SliceThickness
    z_posiciones = [corte[0] for corte in cortes]
    if len(z_posiciones) > 1:
        z_spacing = np.median(np.abs(np.diff(z_posiciones)))
    else:
        z_spacing = 1.0 
        
    espaciado_original = np.array([z_spacing, pixel_spacing[0], pixel_spacing[1]])
    
    return rutas_ordenadas, espaciado_original

Reordenar las vertebras y "normalizar" para reducir el efecto de la distribución binormal de slice thickness y los outliers de "pixel_spacing". Luego la función preprocesar_datase_completo aplica estos cambios a todo el dataset. 

In [8]:
# ==========================================
# PASO 2: ACONDICIONAMIENTO DE LA SEÑAL (HU)
# ==========================================
def procesar_corte_dicom(ruta_dicom, window_center=400, window_width=1800):
    """
    Convierte el píxel crudo a HU y aplica la ventana ósea para resaltar fracturas.
    """
    dcm = pydicom.dcmread(ruta_dicom)
    image = dcm.pixel_array.astype(np.float64)
    
    intercept = getattr(dcm, 'RescaleIntercept', 0)
    slope = getattr(dcm, 'RescaleSlope', 1)
    
    hu_image = image * slope + intercept
    
    img_min = window_center - (window_width / 2) # -500
    img_max = window_center + (window_width / 2) # 1300
    
    windowed_image = np.clip(hu_image, img_min, img_max)
    
    # Normalización Min-Max (0 a 1) para redes neuronales
    img_normalized = (windowed_image - img_min) / (img_max - img_min)
    
    return img_normalized.astype(np.float32)

In [9]:
# ==========================================
# PASO 3: REMUESTREO VOLUMÉTRICO (ISOTRÓPICO)
# ==========================================
def remuestrear_volumen(volumen_3d, espaciado_original, espaciado_objetivo=np.array([1.0, 1.0, 1.0])):
    """
    Homogeneiza el volumen para que 1 vóxel represente 1x1x1 mm.
    """
    factor_redimension = espaciado_original / espaciado_objetivo
    
    # Usar interpolación bilineal/trilineal (order=1) para equilibrar calidad y velocidad
    volumen_remuestreado = scipy.ndimage.zoom(
        volumen_3d, 
        factor_redimension, 
        order=1, 
        mode='nearest'
    )
    
    return volumen_remuestreado

In [10]:
# ==========================================
# PASO 4: ORQUESTACIÓN DEL PIPELINE
# ==========================================
def preprocesar_dataset_completo(path_train, image_dir, output_dir):
    """
    Itera sobre el CSV, ejecuta el preprocesamiento y guarda tensores estandarizados.
    """
    os.makedirs(output_dir, exist_ok=True)
    df_train = pd.read_csv(path_train)
    uids_pacientes = df_train['StudyInstanceUID'].unique()
    
    for uid in tqdm(uids_pacientes, desc="Generando volúmenes preprocesados"):
        rutas_ordenadas, espaciado_original = obtener_rutas_y_espaciado(uid, image_dir)
        
        volumen_procesado = []
        for ruta in rutas_ordenadas:
            img = procesar_corte_dicom(ruta)
            volumen_procesado.append(img)
            
        volumen_3d = np.stack(volumen_procesado)
        
        volumen_isotropico = remuestrear_volumen(volumen_3d, espaciado_original)
        
        # Guardar en disco para el DataLoader de MONAI
        output_path = os.path.join(output_dir, f"{uid}.npz")
        np.savez_compressed(output_path, vol=volumen_isotropico)

#### Entrenamiento Modelos

## 1. Preprocesamiento

In [ ]:
preprocesar_dataset_completo(path_train, image_dir, output_dir)

Generando volúmenes preprocesados:   8%|▊         | 155/2019 [18:28<3:20:57,  6.47s/it]

## 2. Entrenamiento Modelos

### Clasificador Multiclase de Fractura

**Fase 1**

In [ ]:
import os
import glob
import pandas as pd
import torch
from monai.networks.nets import UNet
from monai.losses import DiceLoss
from monai.metrics import DiceMetric
from monai.data import CacheDataset, DataLoader, decollate_batch
from monai.transforms import (
    Compose, LoadImaged, EnsureChannelFirstd, Orientationd,
    Resized, ScaleIntensityRangePercentilesd, AsDiscrete
)

base_dir = "/kaggle/input/competitions/rsna-2022-cervical-spine-fracture-detection"
processed_img_dir = "/kaggle/working/processed_volumes"
seg_dir = os.path.join(base_dir, "segmentations")

df_bbox = pd.read_csv(os.path.join(base_dir, "train_bounding_boxes.csv"))
uids_con_caja = set(df_bbox['StudyInstanceUID'].unique())
uids_seg = [os.path.basename(f).replace(".nii", "") for f in glob.glob(os.path.join(seg_dir, "*.nii"))]

train_files, val_files = [], []
for uid in uids_seg:
    item = {"image": os.path.join(processed_img_dir, f"{uid}.npz"), 
            "label": os.path.join(seg_dir, f"{uid}.nii")}
    if uid in uids_con_caja: val_files.append(item)
    else: train_files.append(item)

# Transformaciones
transforms = Compose([
    LoadImaged(keys=["image", "label"]),
    EnsureChannelFirstd(keys=["image", "label"]),
    Orientationd(keys=["image", "label"], axcodes="RAS"),
    Resized(keys=["image", "label"], spatial_size=(128, 128, 128), mode=("trilinear", "nearest")),
    ScaleIntensityRangePercentilesd(keys="image", lower=1, upper=99, b_min=0.0, b_max=1.0, clip=True)
])

train_loader = DataLoader(CacheDataset(train_files, transform=transforms), batch_size=2, shuffle=True)
val_loader = DataLoader(CacheDataset(val_files, transform=transforms), batch_size=2, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = UNet(spatial_dims=3, in_channels=1, out_channels=8, channels=(16, 32, 64, 128, 256), strides=(2, 2, 2, 2), num_res_units=2).to(device)

loss_function = DiceLoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), 1e-4)
dice_metric = DiceMetric(include_background=False, reduction="mean")

# Entrenamiento simplificado
best_metric = -1
for epoch in range(30):
    model.train()
    for batch_data in train_loader:
        inputs, labels = batch_data["image"].to(device), batch_data["label"].to(device)
        optimizer.zero_grad()
        loss = loss_function(model(inputs), labels)
        loss.backward()
        optimizer.step()
    
    # Validación
    model.eval()
    with torch.no_grad():
        for val_data in val_loader:
            val_inputs, val_labels = val_data["image"].to(device), val_data["label"].to(device)
            val_outputs = [AsDiscrete(argmax=True, to_onehot=8)(i) for i in decollate_batch(model(val_inputs))]
            val_labels = [AsDiscrete(to_onehot=8)(i) for i in decollate_batch(val_labels)]
            dice_metric(y_pred=val_outputs, y=val_labels)
        
        metric = dice_metric.aggregate().item()
        dice_metric.reset()
        if metric > best_metric:
            best_metric = metric
            torch.save(model.state_dict(), "/kaggle/working/best_unet3d.pth")

**Fase 2**

In [ ]:
import numpy as np
import scipy.ndimage
from tqdm import tqdm

cropped_dir = "/kaggle/working/cropped_volumes"
os.makedirs(cropped_dir, exist_ok=True)

# Cargar el modelo entrenado en modo evaluación
model.load_state_dict(torch.load("/kaggle/working/best_unet3d.pth"))
model.eval()

df_train = pd.read_csv(os.path.join(base_dir, "train.csv"))
todos_uids = df_train['StudyInstanceUID'].unique()

inference_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    Resized(keys=["image"], spatial_size=(128, 128, 128), mode="trilinear"),
    ScaleIntensityRangePercentilesd(keys="image", lower=1, upper=99, b_min=0.0, b_max=1.0, clip=True)
])

with torch.no_grad():
    for uid in tqdm(todos_uids, desc="Recortando C1-C7"):
        img_path = os.path.join(processed_img_dir, f"{uid}.npz")
        if not os.path.exists(img_path): continue
            
        # Preparar volumen para inferencia
        data = inference_transforms({"image": img_path})
        input_tensor = data["image"].unsqueeze(0).to(device)
        
        # Predecir máscara (argmax elimina la dimensión de canal)
        pred_mask = torch.argmax(model(input_tensor), dim=1).squeeze().cpu().numpy()
        
        # Encontrar los límites tridimensionales (bounding box) donde hay vértebras (clases 1 a 7)
        nonzero_indices = np.argwhere(pred_mask > 0)
        if len(nonzero_indices) == 0: continue # Falla de segmentación
            
        z_min, y_min, x_min = nonzero_indices.min(axis=0)
        z_max, y_max, x_max = nonzero_indices.max(axis=0)
        
        # Cargar el volumen original de alta resolución
        vol_original = np.load(img_path)['vol']
        
        # Mapear las coordenadas de 128x128x128 al tamaño original
        scale_z = vol_original.shape[0] / 128.0
        scale_y = vol_original.shape[1] / 128.0
        scale_x = vol_original.shape[2] / 128.0
        
        z0, z1 = int(z_min * scale_z), int(z_max * scale_z)
        y0, y1 = int(y_min * scale_y), int(y_max * scale_y)
        x0, x1 = int(x_min * scale_x), int(x_max * scale_x)
        
        # Recortar y guardar
        vol_recortado = vol_original[z0:z1, y0:y1, x0:x1]
        np.savez_compressed(os.path.join(cropped_dir, f"{uid}.npz"), vol=vol_recortado)

**Fase 3**

In [ ]:
from monai.networks.nets import DenseNet121
from monai.transforms import RandAffined, RandGaussianNoised
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import recall_score
import torch.nn as nn

# (Asume que df_merged se creó estratificando por n_slices_q y patient_overall)
# train_df y val_df se construyen a partir del StratifiedGroupKFold

train_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    Resized(keys=["image"], spatial_size=(96, 96, 96)), # Tamaño uniforme para DenseNet
    ScaleIntensityRangePercentilesd(keys="image", lower=1, upper=99, b_min=0.0, b_max=1.0, clip=True),
    RandAffined(keys=["image"], prob=0.5, rotate_range=(0.2, 0.2, 0.2)),
    RandGaussianNoised(keys=["image"], prob=0.1)
])

# Preparar listas de diccionarios para MONAI
def prep_dicts(df):
    return [{"image": f"{cropped_dir}/{r.StudyInstanceUID}.npz", 
             "label": np.array([r.patient_overall, r.C1, r.C2, r.C3, r.C4, r.C5, r.C6, r.C7], dtype=np.float32)} 
            for _, r in df.iterrows() if os.path.exists(f"{cropped_dir}/{r.StudyInstanceUID}.npz")]

train_loader = DataLoader(CacheDataset(prep_dicts(train_df), transform=train_transforms), batch_size=4, shuffle=True)
val_loader = DataLoader(CacheDataset(prep_dicts(val_df), transform=train_transforms), batch_size=4, shuffle=False)

classifier = DenseNet121(spatial_dims=3, in_channels=1, out_channels=8).to(device)

# Configuración de hiper-sensibilidad
pos_counts = train_df[['patient_overall', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7']].values.sum(axis=0)
neg_counts = len(train_df) - pos_counts
penalty_factor = 2.5 
pos_weights = torch.tensor((neg_counts / pos_counts) * penalty_factor, dtype=torch.float32).to(device)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weights)
optimizer = torch.optim.AdamW(classifier.parameters(), lr=1e-4)

DECISION_THRESHOLD = 0.3
best_val_recall = 0.0

for epoch in range(30):
    classifier.train()
    for batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(classifier(batch["image"].to(device)), batch["label"].to(device))
        loss.backward()
        optimizer.step()
        
    classifier.eval()
    val_preds_probs, val_trues = [], []
    with torch.no_grad():
        for batch in val_loader:
            outputs = classifier(batch["image"].to(device))
            val_preds_probs.append(torch.sigmoid(outputs).cpu().numpy())
            val_trues.append(batch["label"].numpy())
            
    val_preds_probs = np.vstack(val_preds_probs)
    val_trues = np.vstack(val_trues)
    
    val_preds_binary = (val_preds_probs > DECISION_THRESHOLD).astype(int)
    recalls = [recall_score(val_trues[:, i], val_preds_binary[:, i], zero_division=0) for i in range(8)]
    mean_recall = np.mean(recalls)
    
    if mean_recall > best_val_recall:
        best_val_recall = mean_recall
        torch.save(classifier.state_dict(), "/kaggle/working/best_recall_classifier.pth")
        print(f"Epoch {epoch+1} | Nuevo mejor Recall: {best_val_recall:.4f}")

**Curva de Aprendizaje**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from monai.networks.nets import DenseNet121
from monai.transforms import RandAffined, RandGaussianNoised
from sklearn.metrics import recall_score
import torch.nn as nn
import numpy as np
import torch

# (Configuración previa de DataLoaders, modelo y pérdida se mantiene igual)

DECISION_THRESHOLD = 0.3
best_val_recall = 0.0
max_epochs = 30

# Listas para almacenar el historial de las curvas de aprendizaje
history_train_loss = []
history_val_loss = []
history_val_recall = []

for epoch in range(max_epochs):
    # ==========================
    # ENTRENAMIENTO
    # ==========================
    classifier.train()
    train_loss = 0.0
    for batch in train_loader:
        optimizer.zero_grad()
        # Asegurarse de enviar las imágenes y etiquetas al dispositivo (GPU/CPU)
        loss = criterion(classifier(batch["image"].to(device)), batch["label"].to(device))
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
        
    avg_train_loss = train_loss / len(train_loader)
    history_train_loss.append(avg_train_loss)
        
    # ==========================
    # VALIDACIÓN
    # ==========================
    classifier.eval()
    val_loss = 0.0
    val_preds_probs, val_trues = [], []
    
    with torch.no_grad():
        for batch in val_loader:
            inputs, labels = batch["image"].to(device), batch["label"].to(device)
            outputs = classifier(inputs)
            
            # Acumular pérdida de validación
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            # Guardar predicciones para métricas
            val_preds_probs.append(torch.sigmoid(outputs).cpu().numpy())
            val_trues.append(labels.cpu().numpy())
            
    avg_val_loss = val_loss / len(val_loader)
    history_val_loss.append(avg_val_loss)
            
    val_preds_probs = np.vstack(val_preds_probs)
    val_trues = np.vstack(val_trues)
    
    # Calcular Recall
    val_preds_binary = (val_preds_probs > DECISION_THRESHOLD).astype(int)
    recalls = [recall_score(val_trues[:, i], val_preds_binary[:, i], zero_division=0) for i in range(8)]
    mean_recall = np.mean(recalls)
    history_val_recall.append(mean_recall)
    
    # Guardar el mejor modelo
    if mean_recall > best_val_recall:
        best_val_recall = mean_recall
        torch.save(classifier.state_dict(), "/kaggle/working/best_recall_classifier.pth")
        estado_modelo = " (Guardado)"
    else:
        estado_modelo = ""
        
    print(f"Epoch {epoch+1:02d} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f} | Val Recall: {mean_recall:.4f}{estado_modelo}")

# ==========================
# VISUALIZACIÓN DE CURVAS
# ==========================
sns.set_theme(style="whitegrid")
plt.figure(figsize=(14, 5))

# Gráfico 1: Train Loss vs Val Loss
plt.subplot(1, 2, 1)
plt.plot(range(1, max_epochs + 1), history_train_loss, label='Train Loss', marker='o', markersize=4)
plt.plot(range(1, max_epochs + 1), history_val_loss, label='Validation Loss', marker='o', markersize=4)
plt.title('Curva de Pérdida (Detectar Sobreajuste)')
plt.xlabel('Épocas')
plt.ylabel('BCEWithLogitsLoss')
plt.legend()

# Gráfico 2: Evolución del Recall
plt.subplot(1, 2, 2)
plt.plot(range(1, max_epochs + 1), history_val_recall, label='Validation Recall', color='green', marker='s', markersize=4)
plt.axhline(y=best_val_recall, color='red', linestyle='--', alpha=0.5, label=f'Best Recall ({best_val_recall:.4f})')
plt.title('Evolución del Recall (Sensibilidad)')
plt.xlabel('Épocas')
plt.ylabel('Recall Medio')
plt.legend()

plt.tight_layout()
plt.show()

Carga de datos, bucle y modelo de entrenamiento

Entrenamiento de modelos